In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

# =========================
# CONFIG
# =========================
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

base_urls = [
    'https://www.detik.com/tag/ekonomi',
    'https://www.detik.com/tag/politik',
    'https://www.detik.com/tag/teknologi',
    'https://www.detik.com/tag/lifestyle',
    'https://www.detik.com/tag/olahraga',
    'https://www.detik.com/tag/kesehatan',
    'https://www.detik.com/tag/pendidikan',
    'https://www.detik.com/tag/lingkungan-hidup',
    'https://www.detik.com/tag/hukum'
]

categories = [
    "Economy",
    "Politics",
    "Technology",
    "Lifestyle",
    "Sports",
    "Health",
    "Education",
    "Environment",
    "Law"
]

# =========================
# AMBIL ISI ARTIKEL
# =========================
def crawl_detik_content(link):
    try:
        resp = requests.get(link, headers=HEADERS, timeout=10)
        if resp.status_code != 200:
            return None

        soup = BeautifulSoup(resp.text, "html.parser")
        content_div = soup.find("div", class_="detail__body-text")

        if not content_div:
            return None

        paragraphs = content_div.find_all("p")
        isi = " ".join(p.text.strip() for p in paragraphs)

        return isi

    except:
        return None


# =========================
# MAIN CRAWLER (RANDOM GLOBAL)
# =========================
def crawl_detik_random_total(
    base_urls,
    categories,
    target_total=1260,
    berita_per_page=10,
    max_page=80,
    output_csv="detik_random_1260.csv"
):
    assert len(base_urls) == len(categories), "Jumlah URL & kategori harus sama"

    data = []
    seen_links = set()

    # shuffle kategori biar random
    combined = list(zip(base_urls, categories))
    random.shuffle(combined)

    print(f" Target total: {target_total}")

    for base_url, category in combined:
        print(f"\n Crawling {category}")

        for page in range(1, max_page + 1):
            url = f"{base_url}?page={page}"
            print(f" {category} - Page {page}")

            try:
                resp = requests.get(url, headers=HEADERS, timeout=10)
                if resp.status_code != 200:
                    continue

                soup = BeautifulSoup(resp.text, "html.parser")
                articles = soup.find_all("article")

                if not articles:
                    break

                articles = articles[:berita_per_page]

                for art in articles:
                    try:
                        a_tag = art.find("a", href=True)
                        if not a_tag:
                            continue

                        link = a_tag["href"]

                        # skip duplicate lebih awal
                        if link in seen_links:
                            continue

                        judul_tag = art.find("h2")
                        judul = judul_tag.text.strip() if judul_tag else None

                        isi = crawl_detik_content(link)
                        if not isi:
                            continue

                        data.append({
                            "judul": judul,
                            "link": link,
                            "isi": isi,
                            "kategori": category
                        })

                        seen_links.add(link)

                        print(f" {len(data)} data terkumpul")

                        # STOP kalau sudah cukup
                        if len(data) >= target_total * 2:
                            break

                        time.sleep(0.3)

                    except:
                        continue

                if len(data) >= target_total * 2:
                    break

            except:
                continue

        if len(data) >= target_total * 2:
            break

    # =========================
    # SAMPLING RANDOM
    # =========================
    df = pd.DataFrame(data)
    df.drop_duplicates(subset="link", inplace=True)

    print(f"\n Total sebelum sampling: {len(df)}")

    if len(df) < target_total:
        print(" Data kurang dari target, pakai semua")
        df_final = df
    else:
        df_final = df.sample(n=target_total, random_state=42)

    df_final.to_csv(output_csv, index=False)

    print("\n SELESAI")
    print(f" Final data: {len(df_final)}")
    print(f" File: {output_csv}")


# =========================
# RUN
# =========================
if __name__ == "__main__":
    crawl_detik_random_total(
        base_urls,
        categories,
        target_total=1260,
        max_page=80
    )

 Target total: 1260

 Crawling Law
 Law - Page 1
 1 data terkumpul
 2 data terkumpul
 3 data terkumpul
 4 data terkumpul
 5 data terkumpul
 6 data terkumpul
 7 data terkumpul
 8 data terkumpul
 9 data terkumpul
 Law - Page 2
 10 data terkumpul
 11 data terkumpul
 12 data terkumpul
 13 data terkumpul
 14 data terkumpul
 15 data terkumpul
 16 data terkumpul
 17 data terkumpul
 18 data terkumpul
 19 data terkumpul
 Law - Page 3
 20 data terkumpul
 21 data terkumpul
 22 data terkumpul
 23 data terkumpul
 24 data terkumpul
 25 data terkumpul
 26 data terkumpul
 27 data terkumpul
 28 data terkumpul
 29 data terkumpul
 Law - Page 4
 30 data terkumpul
 31 data terkumpul
 32 data terkumpul
 33 data terkumpul
 34 data terkumpul
 35 data terkumpul
 36 data terkumpul
 37 data terkumpul
 38 data terkumpul
 Law - Page 5
 39 data terkumpul
 40 data terkumpul
 41 data terkumpul
 42 data terkumpul
 43 data terkumpul
 44 data terkumpul
 45 data terkumpul
 46 data terkumpul
 47 data terkumpul
 48 data te